## Importando dependências

In [2]:
import pandas as pd

In [3]:
#Execuções InOrder
inorder1 = pd.read_csv('InOrder/sim_Sat_Jul_11_14_59_16_2026_.csv')
inorder2 = pd.read_csv('InOrder/sim_Sat_Jul_11_16_24_19_2026_.csv')
inorder3 = pd.read_csv('InOrder/sim_Sat_Jul_11_17_08_00_2026_.csv')

In [4]:
#Execuções OutOfOrder
outoforder1 = pd.read_csv('OutOrder/sim_Sat_Jul_11_18_49_17_2026_.csv')
outoforder2 = pd.read_csv('OutOrder/sim_Sat_Jul_11_19_40_44_2026_.csv')
outoforder3 = pd.read_csv('OutOrder/sim_Sat_Jul_11_21_20_43_2026_.csv')

In [5]:
#Execuções InOrder BPU Adaptive
inOrderBPUAdaptive1 = pd.read_csv('InOrder-Gshare/sim_Sun_Jul_12_16_08_53_2026_.csv')
inOrderBPUAdaptive2 = pd.read_csv('InOrder-Gshare/sim_Sun_Jul_12_16_55_31_2026_.csv')
inOrderBPUAdaptive3 = pd.read_csv('InOrder-Gshare/sim_Sun_Jul_12_18_13_24_2026_.csv')

In [6]:
#Execuções InOrder L2 Cache 512K
inOrderL2512K1 = pd.read_csv('InOrder-512L2/sim_Sun_Jul_12_19_22_02_2026_.csv')
inOrderL2512K2 = pd.read_csv('InOrder-512L2/sim_Sun_Jul_12_20_04_29_2026_.csv')
inOrderL2512K3 = pd.read_csv('InOrder-512L2/sim_Sun_Jul_12_21_01_19_2026_.csv')

## Processamento de dados

In [7]:
inorder1.head()

,stat-name,user,supervisor,hypervisor,machine,total
0,cycles,19449037501,11034907,0,2330904,19462403312
1,commits,15001680643,5576442,0,1573068,15008830153
2,ins_fetch,15764173231,6297704,0,1687653,15772158588
3,sim_time_milli_sec,0,0,0,0,2474732
4,insn_mem_delay,255789,654423,0,3880,914092


In [8]:
valor = inorder1.loc[inorder1['stat-name'] == 'cycles', 'total'].iloc[0]

In [ ]:
def calcula_metricas(df_bruto, seed_name):
    df = df_bruto.set_index('stat-name')
    
    def get_val(stat):
        return df.loc[stat, 'total'] if stat in df.index else 0

    # 1. Cálculos de Instruções e IPC
    ciclos = get_val('cycles')
    instrucoes = get_val('commits') if 'commits' in df.index else get_val('instructions')
    ipc = instrucoes / ciclos if ciclos > 0 else 0

    # 2. Branch Prediction
    cond_branch_correct = get_val('cond_branches_pred_correct')
    cond_branch_incorrect = get_val('cond_branches_pred_incorrect')
    total_branches = cond_branch_correct + cond_branch_incorrect
    cond_branch_accuracy = cond_branch_correct / total_branches if total_branches > 0 else 0

    uncond_branch_correct = get_val('uncond_branches_pred_correct')
    uncond_branch_incorrect = get_val('uncond_branches_pred_incorrect')
    total_unconditional_branches = uncond_branch_correct + uncond_branch_incorrect
    uncond_branch_accuracy = uncond_branch_correct / total_unconditional_branches if total_unconditional_branches > 0 else 0

    # 3. Cache Miss Rates
    l1_icache_reads = get_val('L1_icache_reads')
    l1_icache_read_miss_rate = get_val('L1_icache_read_misses') / l1_icache_reads if l1_icache_reads > 0 else 0

    l1_dcache_reads = get_val('L1_dcache_reads')
    l1_dcache_read_miss_rate = get_val('L1_dcache_read_misses') / l1_dcache_reads if l1_dcache_reads > 0 else 0

    l1_dcache_writes = get_val('L1_dcache_writes')
    l1_dcache_write_miss_rate = get_val('L1_dcache_write_misses') / l1_dcache_writes if l1_dcache_writes > 0 else 0

    l2_cache_reads = get_val('L2_cache_reads')
    l2_cache_read_miss_rate = get_val('L2_cache_read_misses') / l2_cache_reads if l2_cache_reads > 0 else 0

    l2_cache_writes = get_val('L2_cache_writes')
    l2_cache_write_miss_rate = get_val('L2_cache_write_misses') / l2_cache_writes if l2_cache_writes > 0 else 0

    # 4. TLB Miss Rates
    itlb_reads = get_val('itlb_reads')
    itlb_miss_rate = 1.0 - (get_val('itlb_hits') / itlb_reads) if itlb_reads > 0 else 0
    load_tlb_reads = get_val('load_tlb_reads')
    load_tlb_miss_rate = 1.0 - (get_val('load_tlb_hits') / load_tlb_reads) if load_tlb_reads > 0 else 0
    store_tlb_reads = get_val('store_tlb_reads')
    store_tlb_miss_rate = 1.0 - (get_val('store_tlb_hits') / store_tlb_reads) if store_tlb_reads > 0 else 0

    # Dicionário mapeando o Nome da Métrica -> Valor Calculado
    dados_metricas = {
        'IPC': ipc,
        'Cond Branch Accuracy': cond_branch_accuracy,
        'Uncond Branch Accuracy': uncond_branch_accuracy,
        'L1i Cache Read Miss Rate': l1_icache_read_miss_rate,
        'L1d Cache Read Miss Rate': l1_dcache_read_miss_rate,
        'L1d Cache Write Miss Rate': l1_dcache_write_miss_rate,
        'L2 Cache Read Miss Rate': l2_cache_read_miss_rate,
        'L2 Cache Write Miss Rate': l2_cache_write_miss_rate,
        'ITLB Miss Rate': itlb_miss_rate,
        'Load TLB Miss Rate': load_tlb_miss_rate,
        'Store TLB Miss Rate': store_tlb_miss_rate
    }

    return pd.Series(dados_metricas, name=f'metricas_{seed_name}')

In [14]:
s_seed1 = calcula_metricas(inorder1, seed_name='seed1')
s_seed2 = calcula_metricas(inorder2, seed_name='seed2')
s_seed3 = calcula_metricas(inorder3, seed_name='seed3')
df_cenario_inorder = pd.concat([s_seed1, s_seed2, s_seed3], axis=1)
df_cenario_inorder['Média'] = df_cenario_inorder[['metricas_seed1', 'metricas_seed2', 'metricas_seed3']].mean(axis=1)

In [ ]:
s_seed1 = calcula_metricas(outoforder1, seed_name='seed1')
s_seed2 = calcula_metricas(outoforder2, seed_name='seed2')
s_seed3 = calcula_metricas(outoforder3, seed_name='seed3')
df_cenario_outoforder = pd.concat([s_seed1, s_seed2, s_seed3], axis=1)
df_cenario_outoforder['Média'] = df_cenario_outoforder[['metricas_seed1', 'metricas_seed2', 'metricas_seed3']].mean(axis=1)